In [21]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv('Data/covid_toy.csv')

In [3]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [ ]:
df.shape

(100, 6)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [12]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

there is missing values in fever let's handle that first

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [15]:
X_train

,age,gender,fever,cough,city
31,83,Male,103.0,Mild,Kolkata
11,65,Female,98.0,Mild,Mumbai
75,5,Male,102.0,Mild,Kolkata
59,6,Female,104.0,Mild,Kolkata
14,51,Male,104.0,Mild,Bangalore
...,...,...,...,...,...
47,18,Female,104.0,Mild,Bangalore
50,19,Male,101.0,Mild,Delhi
22,71,Female,98.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore


## 1. Let's do without using column transformer class

In [ ]:
# first handle missing values
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

In [27]:
print(X_train_fever.shape)
print(X_test_fever.shape)

(80, 1)
(20, 1)


In [ ]:
# Ordinalencoding -> cough 

oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])


In [29]:
print(X_train_cough.shape)
print(X_test_cough.shape)

(80, 1)
(20, 1)


In [32]:
# OneHotEncoding --> gender, city
ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

In [34]:
print(X_train_gender_city.shape)
print(X_test_gender_city.shape)

(80, 4)
(20, 4)


In [35]:
X_train

,age,gender,fever,cough,city
31,83,Male,103.0,Mild,Kolkata
11,65,Female,98.0,Mild,Mumbai
75,5,Male,102.0,Mild,Kolkata
59,6,Female,104.0,Mild,Kolkata
14,51,Male,104.0,Mild,Bangalore
...,...,...,...,...,...
47,18,Female,104.0,Mild,Bangalore
50,19,Male,101.0,Mild,Delhi
22,71,Female,98.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore


In [ ]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

In [38]:
print(X_train_age.shape)
print(X_test_age.shape)

(80, 1)
(20, 1)


In [39]:

X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

In [40]:
print(X_train_transformed.shape)
print(X_test_transformed.shape)

(80, 7)
(20, 7)


## 2. Using ColumnTransformer

In [42]:
from sklearn.compose import ColumnTransformer

In [43]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [ ]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),['fever']), 
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),#name, process, column name
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender', 'city'])
], remainder='passthrough') # 2 optitions passthorugh and drop

In [46]:
transformer.fit_transform(X_train)
transformer.fit_transform(X_test)

array([[ 99.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  65.        ],
       [102.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  33.        ],
       [101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  42.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  83.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  47.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   8.        ],
       [ 98.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  80.        ],
       [104.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  75.        ],
       [ 99.        ,   1.        ,   1.        ,   0.        ,
          0.    

In [47]:
print(transformer.fit_transform(X_train).shape)
print(transformer.fit_transform(X_test).shape)

(80, 7)
(20, 7)
